In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS 02_global_tech_silver.transformed_tables;

In [0]:
from pyspark.sql.functions import col, trim, to_date, when, monotonically_increasing_id

In [0]:
company_df = spark.table("01_global_tech_bronze.raw_tables.company_raw")

In [0]:
cleaned_company_df = company_df.withColumn("company_name", trim(col("company_name"))) \
    .withColumn("industry", trim(col("industry"))) \
    .withColumn("country", trim(col("country"))) \
    .withColumn("established_date", to_date(col("established_date")))

cleaned_company_df = cleaned_company_df \
    .withColumn("company_name", when(col("company_name").isNull(), "Unknown").otherwise(col("company_name"))) \
    .withColumn("industry", when(col("industry").isNull(), "Unknown").otherwise(col("industry"))) \
    .withColumn("country", when(col("country").isNull(), "Unknown").otherwise(col("country")))

# display(cleaned_company_df)

In [0]:
transformed_company_df = cleaned_company_df.withColumnRenamed("is_active", "company_is_active") \
    .withColumn("company_sk", monotonically_increasing_id()) \
    .select("company_sk", "company_id", "company_name", "industry", "country", "established_date", "company_is_active")

transformed_company_df.write.format("delta").mode("overwrite").saveAsTable("02_global_tech_silver.transformed_tables.transformed_company")

# display(transformed_company_df)